In [0]:
from pyspark.sql.functions import *

claims = spark.table("sdp_catalog.bronze.claims_raw")

claims_clean = claims \
    .dropDuplicates() \
    .withColumn(
        "claim_amount",
        col("claim_amount").cast("double")
    ) \
    .drop("claim_status")

claims_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("sdp_catalog.silver.claims_clean")

In [0]:
customers = spark.table("sdp_catalog.bronze.customers_raw")

customers_clean = customers \
.withColumn(
    "customer_name",
    upper(col("customer_name"))
)

customers_clean.write.format("delta") \
.mode("overwrite") \
.saveAsTable("sdp_catalog.silver.customers_clean")

In [0]:
policies = spark.table("sdp_catalog.bronze.policies_raw")

policies_clean = policies.filter(
    col("premium_amount") > 0
)

policies_clean.write.format("delta") \
.mode("overwrite") \
.saveAsTable("sdp_catalog.silver.policies_clean")

In [0]:
%sql
describe history sdp_catalog.silver.policies_clean

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-05-11T05:22:19.000Z,78349980997104,cboxesch@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.checkpoint.writeStatsAsJson"":""false"",""delta.checkpoint.writeStatsAsStruct"":""true"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2819117789394222),0fa69c70-2cf6-410a-9689-d23175c1ce6a,0511-045124-wk8i7nwd-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 7, numOutputBytes -> 1732)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-05-11T03:13:33.000Z,78349980997104,cboxesch@gmail.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.checkpoint.writeStatsAsJson"":""false"",""delta.checkpoint.writeStatsAsStruct"":""true"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)","List(, null, null, , null, null)",null,01f14ce7-629f-156b-bd64-90282dcc2ff0,null,null,WriteSerializable,true,"Map(numFiles -> 0, numOutputRows -> 0, numOutputBytes -> 0)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT * FROM sdp_catalog.silver.policies_clean VERSION AS OF 1

policy_id,policy_type,premium_amount,start_date,end_date
P001,Auto,1200.0,2025-01-01,2026-01-01
P002,Salud,2500.0,2025-01-01,2026-01-01
P003,Hogar,900.0,2025-01-01,2026-01-01
P004,Viaje,300.0,2025-01-01,2026-01-01
P005,Empresarial,10000.0,2025-01-01,2026-01-01
P006,Hogar,850.0,2025-01-01,2026-01-01
P007,Auto,1500.0,2025-01-01,2026-01-01
